# SCADA Diagnostic Console — demo

Alarm → root cause → recommended fix, on multivariate machines (MetroPT-3 compressor, C-MAPSS turbofan).

This notebook runs on the synthetic fixtures so it needs no large downloads. Drop the real files in `data/raw/` to use field data (see `scripts/download_data.py`).

In [ ]:
# Generate synthetic fixtures if not present
import os, subprocess, sys
if not os.path.exists('../data/raw/_fixtures/MetroPT3.csv'):
    subprocess.run([sys.executable, '../scripts/make_demo_fixtures.py'], check=True)

In [ ]:
# Load the MetroPT-3 compressor, detect alarms, diagnose the first one
import sys; sys.path.insert(0, '..')
from src.metropt_preprocessing import load_metropt
from src.diagnosis import extract_alarms, diagnose

b = load_metropt()
res = extract_alarms(b, percentile=99, persistence=3)
print(f'{b.dataset} ({b.source}): {len(res["events"])} alarm(s), threshold={res["threshold"]:.2f}')

diag = diagnose(b, res['events'][0])
print('\nALARM at', diag.alarm.time, '| severity:', diag.alarm.severity)
print('PROBABLE CAUSE:', diag.cause)
print('SIGNATURE:', diag.signature, '| confidence:', diag.confidence)
print('\nTop contributors:')
for c in diag.contributions[:5]:
    print(f'  {c.label:24s} {c.z:+.2f} sigma  {c.direction:5s}  {c.pct:.0f}%')
print('\nRECOMMENDED FIX:', diag.recommendation)

## Launch the interactive console

Run the line below in a terminal (not inline) and open http://127.0.0.1:8050

```bash
python -m src.diagnostic_console
```